# 12 Data Transformation — Every Transformation as Issue → Detect → Fix → Verify

This notebook is a practical, executable lesson. Each step follows **Context → Source Problem → Detect → Fix → Verify → Business Result → Exercise**.

## 1. Standardization — dates

Issue: sources may contain DD-MM-YYYY, MM/DD/YYYY and YYYY-MM-DD. Fix: parse using source-specific rules and produce canonical YYYY-MM-DD. Never silently reinterpret ambiguous dates.

In [ ]:
import pandas as pd
dates=pd.read_csv('data/date_standardization_test_cases.csv')
print(dates)
# Example: explicit source rule for DD-MM-YYYY
dates['parsed_ddmmyyyy']=pd.to_datetime(dates['raw_date'],format='%d-%m-%Y',errors='coerce')
print(dates[['raw_date','parsed_ddmmyyyy']])

## 2. Standardization — text, codes and categories

Issue: `Pune`, ` pune ` and `PUNE` fail equality tests. Fix: trim, normalize case, then map to controlled values.

In [ ]:
df=pd.DataFrame({'city':['Pune',' pune ','PUNE','Nashik']})
df['city_std']=df['city'].str.strip().str.title()
print(df)

## 3. Validation

Issue: quantity, price or discount may be outside allowed ranges. Detect with boolean rules and separate invalid rows.

In [ ]:
sales=pd.read_csv('../11_staging_layer/data/stg_sales.csv')
invalid=sales[(sales['quantity']<=0)|(sales['unit_price']<=0)|(sales['discount_rate']<0)|(sales['discount_rate']>1)]
print('invalid rows:',len(invalid))

## 4. Derivation and calculation

Issue: source may not contain gross amount or margin. Derive them using business rules.

In [ ]:
sales['gross_amount']=sales['quantity']*sales['unit_price']
sales['net_amount']=sales['gross_amount']*(1-sales['discount_rate'])
sales['margin_pct']=((sales['net_amount']-sales['cost_amount'])/sales['net_amount']).replace([float('inf'),-float('inf')],0)
print(sales[['quantity','unit_price','discount_rate','gross_amount','net_amount','margin_pct']].head())

## 5. Lookup enrichment

Issue: transactions contain codes but management needs descriptions. Use reference/master lookups without creating dimensions yet.

In [ ]:
products=pd.read_csv('../11_staging_layer/data/stg_product.csv')
sales=sales.merge(products[['product_id','product_name','category']],on='product_id',how='left',validate='many_to_one')
print(sales[['product_id','product_name','category']].head())

## 6. Deduplication

Issue: duplicate transaction IDs can double revenue. Detect and keep a documented rule.

In [ ]:
dups=sales[sales.duplicated('transaction_id',keep=False)]
print('duplicate transaction records:',len(dups))
sales=sales.drop_duplicates('transaction_id',keep='last')

## 7. Aggregation

Issue: operational transactions are too granular for some reporting needs. Aggregate by business grain.

In [ ]:
monthly=sales.groupby('date_id',as_index=False).agg(total_sales=('sales_amount','sum'),transactions=('transaction_id','nunique'))
monthly.head()

## 8. Pivot / unpivot

Issue: spreadsheets may store categories as columns. Convert to analysis-friendly long format.

In [ ]:
wide=pd.DataFrame({'store':['S1','S2'],'Vanilla':[100,120],'Chocolate':[90,130]})
long=wide.melt(id_vars='store',var_name='product',value_name='sales')
print(long)

## 9. Pandas transformation

Pandas is the baseline classroom implementation.

In [ ]:
result=sales.groupby('product_id',as_index=False)['sales_amount'].sum().sort_values('sales_amount',ascending=False)
result.head()

## 10. Polars transformation

Polars provides a second implementation and helps students compare APIs and performance.

In [ ]:
import polars as pl
pl_sales=pl.from_pandas(sales)
polars_result=pl_sales.group_by('product_id').agg(pl.col('sales_amount').sum().alias('total_sales')).sort('total_sales',descending=True)
print(polars_result.head())

## 11. DuckDB transformation

DuckDB lets students perform transformations with SQL directly against files/dataframes.

In [ ]:
import duckdb
q=duckdb.sql("SELECT product_id, SUM(sales_amount) total_sales FROM sales GROUP BY product_id ORDER BY total_sales DESC LIMIT 10")
print(q)

## 12. Verify transformation

Every transformation needs reconciliation.

In [ ]:
source_total=sales['sales_amount'].sum()
assert source_total>=0
print('transformed row count:',len(sales),'sales:',source_total)

## Exercises

1. Run every code cell and inspect the output.
2. Modify at least one input value to create a data problem.
3. Detect the problem with code.
4. Fix it without hiding the original record.
5. Verify the result and explain the business impact.